<a href="https://colab.research.google.com/github/gulsum135/Datathon2026/blob/main/datathon2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv
Saving test_x.csv to test_x.csv


In [ ]:
import os
import pandas as pd

# Bulunduğumuz klasördeki dosyaları listeyelim
print("Yüklenen Dosyalar:", os.listdir())

# Eğer listede train.csv görüyorsan okumayı dene:
try:
    train_df = pd.read_csv('train.csv')
    print("Harika! train.csv başarıyla okundu. Boyut:", train_df.shape)
except Exception as e:
    print("Hata oluştu:", e)

Yüklenen Dosyalar: ['.config', 'test_x.csv', 'sample_submission.csv', 'sample_data']
Hata oluştu: [Errno 2] No such file or directory: 'train.csv'


In [ ]:


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# datathon klasörünün içindeki dosyaları listeler
klasor_yolu = '/content/drive/MyDrive/datathon'
print("Klasördeki Dosyalar:", os.listdir(klasor_yolu))

Klasördeki Dosyalar: ['train.csv', 'test_x.csv', 'sample_submission.csv']


In [ ]:
import pandas as pd
import os

klasor_yolu = '/content/drive/MyDrive/datathon'

# Dosyaları tam yollarıyla okuyoruz
train_df = pd.read_csv(os.path.join(klasor_yolu, 'train.csv'))
test_df = pd.read_csv(os.path.join(klasor_yolu, 'test_x.csv'))

print("Başarılı! Veri setleri yüklendi.")
print("Train (Eğitim) Boyutu:", train_df.shape)
print("Test Boyutu:", test_df.shape)

Başarılı! Veri setleri yüklendi.
Train (Eğitim) Boyutu: (10000, 47)
Test Boyutu: (10000, 46)


In [ ]:
# ============================================================
# PART 3 — MODEL EĞİTİMİ (LIGHTGBM REGRESSION)
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Hedef Değişken (Y) ve Özelliklerin (X) Ayrılması
target_col = 'career_success_score'

# Eğer bu hücreyi daha önce çalıştırdıysan veya hata aldıysan y'yi sağlama alalım
if target_col in train_df.columns:
    y = train_df[target_col]
else:
    # Eğer yanlışlıkla train_df içinden target_col düşürüldüyse hata vermemesi için koruma
    raise ValueError(f"⚠️ '{target_col}' sütunu train_df içinde bulunamadı! Lütfen verileri okuttuğun ilk hücreden itibaren sırayla tekrar çalıştır.")

# Modelin eğitimine girmeyecek, sadece satır belirten veya zaten sildiğimiz metin sütunlarını ayıklayalım
gereksiz_sutunlar = [target_col, 'id', 'ID', 'Student_ID', 'student_id', 'mentor_feedback_text', 'mentor_feedback_clean']

# Sadece tablomuzda o an gerçekten var olan sütunları düşüyoruz (KeyError önlemi)
X = train_df.drop(columns=[col for col in gereksiz_sutunlar if col in train_df.columns])

# Test setini de tamamen Train setindeki sütun sıralamasıyla eşitliyoruz
X_test = test_df[X.columns]

print(f"📊 Eğitim Özellik Boyutu (X): {X.shape}")
print(f"🎯 Hedef Değişken Boyutu (y): {y.shape}")
print(f"📝 Test Özellik Boyutu (X_test): {X_test.shape}")

# 2. Train - Validation (Eğitim ve Doğrulama) Ayrımı
# Verinin %20'sini modelin performansını test etmek (validation) için ayırıyoruz
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n🚀 LightGBM Modeli Eğitiliyor...")

# 3. Model Parametrelerinin Tanımlanması (Yarışma metriği MSE için ayarlandı)
model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',             # RMSE (Kök MSE) düşerse MSE de düşer
    n_estimators=1500,         # Maksimum ağaç sayısı
    learning_rate=0.03,        # Adım büyüklüğü (Daha kararlı öğrenme için biraz düşürdüm)
    num_leaves=31,             # Ağaç karmaşıklığı
    random_state=42,
    n_jobs=-1                  # Tüm işlemci çekirdeklerini sonuna kadar kullan
)

# 4. Modelin Eğitilmesi
# Early stopping ile model validation setinde gelişmeyi bıraktığı an eğitimi durdurur (Overfitting önlemi)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(100)]
)

# 5. Model Performansının Ölçülmesi
y_pred_val = model.predict(X_val)
val_mse = mean_squared_error(y_val, y_pred_val)

print("\n========================================================")
print(f"🏆 VALIDATION MSE SKORUN: {val_mse:.4f}")
print(f"📈 VALIDATION RMSE SKORUN: {np.sqrt(val_mse):.4f}")
print("========================================================")

📊 Eğitim Özellik Boyutu (X): (10000, 20054)
🎯 Hedef Değişken Boyutu (y): (10000,)
📝 Test Özellik Boyutu (X_test): (10000, 20054)

🚀 LightGBM Modeli Eğitiliyor...


LightGBMError: Do not support special JSON characters in feature name.

In [ ]:
# ============================================================
# PART 3 — MODEL EĞİTİMİ (LIGHTGBM REGRESSION)
# ============================================================
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import re # Add regex library for cleaning column names

# 1. Hedef Değişken (Y) ve Özelliklerin (X) Ayrılması
target_col = 'career_success_score'

# Eğer bu hücreyi daha önce çalıştırdıysan veya hata aldıysan y'yi sağlama alalım
if target_col in train_df.columns:
    y = train_df[target_col]
else:
    # Eğer yanlışlıkla train_df içinden target_col düşürüldüyse hata vermemesi için koruma
    raise ValueError(f"⚠️ '{target_col}' sütunu train_df içinde bulunamadı! Lütfen verileri okuttuğun ilk hücreden itibaren sırayla tekrar çalıştır.")

# Modelin eğitimine girmeyecek, sadece satır belirten veya zaten sildiğimiz metin sütunlarını ayıklayalım
gereksiz_sutunlar = [target_col, 'id', 'ID', 'Student_ID', 'student_id', 'mentor_feedback_text', 'mentor_feedback_clean']

# Sadece tablomuzda o an gerçekten var olan sütunları düşüyoruz (KeyError önlemi)
X = train_df.drop(columns=[col for col in gereksiz_sutunlar if col in train_df.columns])

# Test setini de tamamen Train setindeki sütun sıralamasıyla eşitliyoruz
X_test = test_df[X.columns]

# ============================================================
# LightGBM uyumlu sütun adlarını temizle
# ============================================================
def clean_col_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        # Özel karakterleri alt çizgi ile değiştir ve birden fazla alt çizgiyi tekine indirge
        new_col = re.sub(r'[^A-z0-9_]', '_', col) # Sadece alfanumerik ve alt çizgiye izin ver
        new_col = re.sub(r'__+', '_', new_col) # Birden fazla alt çizgiyi tekine indirge
        new_col = new_col.strip('_') # Başındaki/sonundaki alt çizgileri sil
        new_cols.append(new_col)
    df.columns = new_cols
    return df

X = clean_col_names(X)
X_test = clean_col_names(X_test)

print(f"📊 Eğitim Özellik Boyutu (X): {X.shape}")
print(f"🎯 Hedef Değişken Boyutu (y): {y.shape}")
print(f"📝 Test Özellik Boyutu (X_test): {X_test.shape}")

# 2. Train - Validation (Eğitim ve Doğrulama) Ayrımı
# Verinin %20'sini modelin performansını test etmek (validation) için ayırıyoruz
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n🚀 LightGBM Modeli Eğitiliyor...")

# 3. Model Parametrelerinin Tanımlanması (Yarışma metriği MSE için ayarlandı)
model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',             # RMSE (Kök MSE) düşerse MSE de düşer
    n_estimators=1500,         # Maksimum ağaç sayısı
    learning_rate=0.03,        # Adım büyüklüğü (Daha kararlı öğrenme için biraz düşürdüm)
    num_leaves=31,             # Ağaç karmaşıklığı
    random_state=42,
    n_jobs=-1                  # Tüm işlemci çekirdeklerini sonuna kadar kullan
)

# 4. Modelin Eğitilmesi
# Early stopping ile model validation setinde gelişmeyi bıraktığı an eğitimi durdurur (Overfitting önlemi)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(100)]
)

# 5. Model Performansının Ölçülmesi
y_pred_val = model.predict(X_val)
val_mse = mean_squared_error(y_val, y_pred_val)

print("\n========================================================")
print(f"🏆 VALIDATION MSE SKORUN: {val_mse:.4f}")
print(f"📈 VALIDATION RMSE SKORUN: {np.sqrt(val_mse):.4f}")
print("========================================================")

📊 Eğitim Özellik Boyutu (X): (10000, 20054)
🎯 Hedef Değişken Boyutu (y): (10000,)
📝 Test Özellik Boyutu (X_test): (10000, 20054)

🚀 LightGBM Modeli Eğitiliyor...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005571 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6271
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 54
[LightGBM] [Info] Start training from score 76.951848
[100]	valid_0's rmse: 10.0175
[200]	valid_0's rmse: 9.50364
[300]	valid_0's rmse: 9.3721
[400]	valid_0's rmse: 9.34057

🏆 VALIDATION MSE SKORUN: 87.2050
📈 VALIDATION RMSE SKORUN: 9.3384


In [ ]:
# ============================================================
# HÜCRE 1 — Kütüphaneler
# ============================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# ============================================================
# HÜCRE 2 — Model parametreleri
# ============================================================
LGBM_PARAMS = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "n_estimators"     : 1000,
    "learning_rate"    : 0.05,
    "num_leaves"       : 64,
    "max_depth"        : -1,
    "min_child_samples": 20,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 1.0,
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbose"          : -1,
}

N_FOLDS   = 5
SEED      = 42
EARLY_STOP= 50   # validation kaybı durduğunda erken dur

# ============================================================
# HÜCRE 3 — 5-Fold CV eğitim döngüsü
# ============================================================
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_tahminler  = np.zeros(len(X))          # Out-of-Fold tahminleri
test_tahminler = np.zeros(len(X_test))     # Her fold tahminlerinin birikimi
fold_mse_listesi = []

print("=" * 60)
print(f"  LightGBM — {N_FOLDS}-Fold Cross Validation")
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**LGBM_PARAMS)

    model.fit(
        X_tr, y_tr,
        eval_set              = [(X_val, y_val)],
        callbacks             = [
            lgb.early_stopping(EARLY_STOP, verbose=False),
            lgb.log_evaluation(period=-1),   # fold içi logu kapat
        ],
    )

    # Validation tahmini (OOF)
    val_pred = model.predict(X_val, num_iteration=model.best_iteration_)
    oof_tahminler[val_idx] = val_pred

    # Test tahmini — 5 fold ortalaması için biriktir
    test_tahminler += model.predict(X_test, num_iteration=model.best_iteration_) / N_FOLDS

    fold_mse = mean_squared_error(y_val, val_pred)
    fold_mse_listesi.append(fold_mse)

    print(f"  Fold {fold}  |  Best iter: {model.best_iteration_:4d}"
          f"  |  Val MSE: {fold_mse:.6f}  |  Val RMSE: {np.sqrt(fold_mse):.6f}")

# ============================================================
# HÜCRE 4 — Özet istatistikler
# ============================================================
oof_mse  = mean_squared_error(y, oof_tahminler)
oof_rmse = np.sqrt(oof_mse)

print("\n" + "=" * 60)
print("  Cross Validation Sonuçları")
print("=" * 60)
print(f"  Fold MSE'leri  : {[round(m, 6) for m in fold_mse_listesi]}")
print(f"  Ortalama MSE   : {np.mean(fold_mse_listesi):.6f}"
      f"  ± {np.std(fold_mse_listesi):.6f}")
print(f"  OOF MSE        : {oof_mse:.6f}")
print(f"  OOF RMSE       : {oof_rmse:.6f}")
print("=" * 60)

# ============================================================
# HÜCRE 5 — Kaggle submission dosyası
# ============================================================

# Olası ID sütun isimlerini test_df içinde arayalım
olasi_id_isimleri = ["id", "ID", "Student_ID", "student_id"]
id_sutunu = None

for col in olasi_id_isimleri:
    if col in test_df.columns:
        id_sutunu = col
        break

if id_sutunu is None:
    # Eğer One-Hot Encoding veya işlemler sırasında ID silindiyse,
    # orijinal test dosyasını Drive'dan hızlıca çekip ID'yi oradan alıyoruz
    orijinal_test = pd.read_csv('/content/drive/MyDrive/datathon/test_x.csv')
    id_verisi = orijinal_test.iloc[:, 0] # İlk sütun genelde ID'dir
    id_sutun_adi = orijinal_test.columns[0]
else:
    id_verisi = test_df[id_sutunu]
    id_sutun_adi = id_sutunu

# Yarışma formatına birebir uygun DataFrame oluşturma
submission = pd.DataFrame({
    id_sutun_adi: id_verisi,
    "career_success_score": test_tahminler  # Yarışmanın beklediği tam hedef sütun adı
})

# Dosyayı hem Google Drive'ına hem de yerel Colab hafızasına kaydedelim (Kayıp olmasın)
submission.to_csv("submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/datathon/final_submission.csv", index=False)

print(f"✅ Dosyalar Başarıyla Kaydedildi!")
print(f"📁 Colab Hafızası: submission.csv")
print(f"📁 Google Drive: /datathon/final_submission.csv")
print(f"📐 Boyut: {submission.shape}")
print("\n📋 İlk 5 Satır Kontrolü:")
print(submission.head())

  LightGBM — 5-Fold Cross Validation
  Fold 1  |  Best iter:  202  |  Val MSE: 88.432624  |  Val RMSE: 9.403862
  Fold 2  |  Best iter:  238  |  Val MSE: 90.393787  |  Val RMSE: 9.507565
  Fold 3  |  Best iter:  213  |  Val MSE: 78.835361  |  Val RMSE: 8.878928
  Fold 4  |  Best iter:  257  |  Val MSE: 79.429309  |  Val RMSE: 8.912312
  Fold 5  |  Best iter:  202  |  Val MSE: 88.811672  |  Val RMSE: 9.423994

  Cross Validation Sonuçları
  Fold MSE'leri  : [88.432624, 90.393787, 78.835361, 79.429309, 88.811672]
  Ortalama MSE   : 85.180551  ± 4.985520
  OOF MSE        : 85.180551
  OOF RMSE       : 9.229331
✅ Dosyalar Başarıyla Kaydedildi!
📁 Colab Hafızası: submission.csv
📁 Google Drive: /datathon/final_submission.csv
📐 Boyut: (10000, 2)

📋 İlk 5 Satır Kontrolü:
   student_id  career_success_score
0  STU_010001             59.432664
1  STU_010002             68.167840
2  STU_010003             72.132729
3  STU_010004             94.597519
4  STU_010005             77.810477
